# Fine-tuning a Pre-trained Convolutional Neural Network for the Grainset Image Dataset

For our Neural Network, we will use a pre-trained model in order to perform classification tasks. We chose a CNN model, specifically EfficientNetV2-small, in order to utilize its convolutional blocks that can extract features from images and is still a relatively small model.

In [ ]:
import matplotlib.pyplot as plt
from neural_network_utils import (
    GrainDataModule,
    IDX_TO_CLASS,
    LightningModel,
    plot_loss_and_acc,
)
import lightning as L
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
from lightning.pytorch.loggers import CSVLogger
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
%load_ext watermark
%watermark -p torch,lightning,torchvision,torchmetrics


## Preparing the Pre-Trained Model

We will use the EfficientNetV2-small model pretrained using the Imagenet1K-v1 dataset in order to leverage the feature extraction ability that the model has acquired when training the model on the thousand classes of images on the dataset.

In [ ]:
efficientnet_v2_s = torch.hub.load(
    "pytorch/vision:v0.25.0", "efficientnet_v2_s", weights="IMAGENET1K_V1"
)

In [ ]:
efficientnet_v2_s

In [ ]:
len(list(efficientnet_v2_s.features))

For this project, we will not train the whole model as it will be computationally expensive. Instead we will train the last 3 convolutional blocks in order to for those blocks to adapt to our dataset. This will also leverage the capability of the earlier convolutional blocks of EfficientNet, to extract the generic features of an image.

Reference: https://www.identifyshell.org/blog-fine-tuning-efficientnetv2-models.php

In [ ]:
for param in efficientnet_v2_s.features[:5].parameters():
    param.requires_grad = False

for param in efficientnet_v2_s.features[5:].parameters():
    param.requires_grad = True

efficientnet_v2_s.classifier[1] = torch.nn.Linear(1280, 14)

In [ ]:
efficientnet_v2_s

In [ ]:
from torchvision.models import EfficientNet_V2_S_Weights

weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1
preprocess_transform = weights.transforms()
preprocess_transform

We will also add some augmentations for the training set. A weighted sampler for the training set is also implemented in the DataModule in order to mitigate the class imbalance in the dataset.

In [ ]:
import torchvision.transforms as T

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
train_transform = T.Compose(
    [
        T.RandomResizedCrop(384),
        T.RandomHorizontalFlip(),
        T.RandomRotation(20),
        T.ColorJitter(0.2, 0.2, 0.2),
        T.ToTensor(),
        T.Normalize(mean, std),
        T.RandomErasing(p=0.25),
    ]
)

## Preparing the Dataset and Loaders
A quick check in order to make sure that the dataloader is able to load the images properly

In [ ]:
data_module = GrainDataModule(
    data_dir="dataset/images",
    batch_size=64,
    train_transform=train_transform,
    val_transform=preprocess_transform,
    test_transform=preprocess_transform,
)

In [ ]:
data_module.setup()

In [ ]:
print("Train size:", len(data_module.train_dataset))
print("Val size:", len(data_module.val_dataset))
print("Test size:", len(data_module.test_dataset))

We used a weighted sampling in order to address the class imbalance in the dataset. Let's check if it working and not sampling the majority classes which are rice_0_NOR and maize_0_nor.

In [ ]:
train_loader = data_module.train_dataloader()

batch_images, batch_labels = next(iter(train_loader))

print(batch_images.shape)
print(batch_labels.shape)

# Display all images in the batch in a grid
n_images = batch_images.shape[0]
n_cols = 8
n_rows = (n_images + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 2 * n_rows))
axes = axes.flatten()

for idx in range(n_images):
    img = batch_images[idx].permute(1, 2, 0)
    img = (img - img.min()) / (img.max() - img.min() + 1e-5)

    axes[idx].imshow(img.cpu().numpy())
    class_label = IDX_TO_CLASS[batch_labels[idx].item()]
    axes[idx].set_title(class_label, fontsize=8)
    axes[idx].axis("off")

for idx in range(n_images, len(axes)):
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

print(f"Displayed {n_images} images from the batch")

# Calculate class distribution
print("CLASS DISTRIBUTION IN BATCH")

class_counts = {}
for label_idx in batch_labels:
    class_name = IDX_TO_CLASS[label_idx.item()]
    class_counts[class_name] = class_counts.get(class_name, 0) + 1

# Sort by counts descending
sorted_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)

for class_name, count in sorted_classes:
    percentage = (count / n_images) * 100
    print(f"{class_name}: {count} images ({percentage:.1f}%)")

# Preparing Hyperparameters for the Model

We will use the utility tuners that come with Lightning in order to find a good baseline for the learning rate and batch sizes.

In [ ]:
L.seed_everything(67)

num_epochs = 1
num_steps = num_epochs * len(data_module.train_dataloader())
lightning_model = LightningModel(
    model=efficientnet_v2_s,
    learning_rate=0.001,
    num_classes=14,
    cosine_t_max=num_steps,
)

debug_trainer = L.Trainer(
    accelerator="gpu",
    devices="auto",
    logger=None,
    deterministic=True,
    precision="f16-mixed",
)

In [ ]:
from lightning.pytorch.tuner import Tuner

In [ ]:
tuner = Tuner(debug_trainer)

new_batch_size = tuner.scale_batch_size(
    model=lightning_model, datamodule=data_module, mode="power"
)

print(new_batch_size)

In [ ]:
lr_finder = tuner.lr_find(lightning_model, datamodule=data_module)

# Plot
fig = lr_finder.plot(suggest=True)
fig.show()

# Get suggested LR
new_lr = lr_finder.suggestion()
print(new_lr)

## Traing the Model

We will use CosineAnnealing learning rate scheduler alongside Adam optimizer in order to improve the loss convergence of the model

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint

callbacks = [
    ModelCheckpoint(save_top_k=1, mode="max", monitor="val_macro_f1", save_last=True)
]

In [ ]:
lightning_model.learning_rate = new_lr

In [ ]:
trainer = L.Trainer(
    max_epochs=num_epochs,
    accelerator="gpu",
    devices="auto",
    logger=CSVLogger(save_dir="logs/", name="tuned-efficientnet"),
    deterministic=True,
    precision="f16-mixed",
    callbacks=callbacks,
)

In [ ]:
trainer.fit(model=lightning_model, datamodule=data_module)

In [ ]:
trainer.test(model=lightning_model, datamodule=data_module)

In [ ]:
plot_loss_and_acc(log_dir=trainer.logger.log_dir, loss_ylim=(0.0, 2.0))